# 리포트 49 — 검출기가 실제로 쓰는 커널 그대로 모호함수를 그렸다

> ### 한 일
> **기준신호 하나가 거리-도플러 평면에 만드는 응답을 검출기와 같은 커널로 계산하고, 검출기의 거리도플러 출력과 대조해 두 값의 최대 편차를 쟀다.**

### 결과
1. 모호함수와 검출기 거리도플러 출력은 최대 0.144 dB [^1] 안에서 같다 (6 [^2]개 경우, −45 dB 이상 셀).
2. 거리 주엽의 **−3 dB 전폭**은 $c/B_{ref}$ 의 89% [^3] (WiFi) · 92% [^4] (LTE) · 94% [^5] (5G) 인데, 이 눈금은 이상적 평탄 스펙트럼도 88.6% 로 읽는 **혼합 규약**이다 — 같은 첫 널 규약끼리 맞대면 1.009(WiFi) · 1.080(LTE) · 1.116(5G) 배다.
3. 도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 [^6]배 근처이고, 이 배수는 파형이 아니라 slow-time Hann 창이 정한다(`src/passive_process.py:142`).
4. **검출기 거리창**(`benchmark/geometry.py:143` `RB_WINDOW_M = 60.0`, |Rb| ≤ 60 m) 안의 2D 부엽 최대는 WiFi -23.4 [^7] · LTE -15.0 [^8] · 5G -18.3 dB [^9] 다 — 지연축을 프레임 전체로 열면 세 값의 순위가 바뀐다. 레플리카는 WiFi -0.00 [^10] · LTE -23.27 dB [^11] 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 커널 | 검출기가 쓰는 것과 **같은 커널**로 계산한다 — `benchmark/verify_ambiguity.py:150`, 검출기는 `src/passive_process.py:133` |
| 대조 방식 | 표준 × 점유 경우마다 −45 dB 이상 셀의 최대 편차를 재고 그 최대값을 싣는다 |
| 부엽을 재는 거리창 | `psl_chamber_db`·`isl_chamber_db` 는 \|Rb\| ≤ 60 m 안에서 잡는다 — 검출기가 실제로 세우는 RD 맵의 거리축이 `benchmark/geometry.py:143` `RB_WINDOW_M = 60.0` → `src/passive_process.py:141` 의 `RP[:, :n_range]` 로 Rb ∈ [0, 60) m 이기 때문이다 |
| 전역 `psl_2d_db` 를 본문 결과에 안 싣는 이유 | 그것은 프레임 한 주기 전체(km 단위 순환 지연축)에서 잡은 값이라 검출기 화면 밖의 봉우리가 정한다 — `benchmark/verify_ambiguity.py:286` 의 주석이 그렇게 적어 뒀다. 표에는 두 창을 나란히 싣는다 |
| 슬로타임 창 | 프레임과 프레임 사이 축에 Hann 창을 씌운다 — 도플러 주엽의 배수를 정하는 것이 이 창이다(`src/passive_process.py:142`) |
| 이 표의 PRF | **검출기 프레임률**이다. 물리 주기 기준의 접힘은 따로 잰다 |
| 기준신호를 어디서 얻나 | 이 커널은 기준신호로 **송신 파형 그 자체**를 넣는다 — 잡음도 다중경로도 없는 «완벽한 기준안테나» 상한이다. 그 가정을 푸는 것은 이 편의 밖이다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json`, `outputs/report03_illuminators.json` |
| 소요 | ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | $\Delta R_b$ 와 잡음대역 규약 |

---

## 모호함수는 검출기의 눈이다

모호함수 $\chi(\tau, f_d)$ 는 기준신호 하나가 거리-도플러 평면에 만드는 응답이다. 표적이 점 하나여도 검출기 화면에는 이 모양이 찍힌다.

우리가 그리는 것은 **검출기가 쓰는 것과 같은 커널**이고, 검출기의 거리도플러 출력과 최대 0.144 dB [^1] (6 [^2]개 경우, −45 dB 이상 셀) 안에서 같다. 따로 계산한 그림이 아니라 **검출기 자신의 눈**이라는 뜻이다.

⚠ 그 눈이 드는 기준신호는 **송신 파형 그 자체**다 — 잡음 0 · 다중경로 0, 즉 기준안테나가 완벽하다는 상한이다. 기준채널을 현실로 두면 그 사슬이 얼마를 잃는지는 ⛔«기준채널이 현실이면 얼마를 잃는가» 별편(2026-09-03 내림 — 동작점이 실내 통제 기하다, `archive/chamber_0903/`) 가 단일축으로 잰다.

## 주엽 — 닫힌형과 대조

거리 주엽(응답에서 가장 높이 솟은 가운데 봉우리)의 **−3 dB 전폭**은 $c/B_{ref}$ 의 89% [^3] (WiFi) · 92% [^4] (LTE) · 94% [^5] (5G) 다.

⚠ **이 비는 규약이 섞여 있다.** 분자는 −3 dB 전폭이고, 분모 $c/B_{ref}$ 는 첫 널 간격이다(편 47 의 규약). 스펙트럼이 완전히 평탄한 이상적 파형도 이 눈금에서는 88.6%($|\mathrm{sinc}|$ 의 −3 dB 전폭)로 읽힌다. 세 값에서 그 바닥을 뺀 파형 고유분은 +1.0% · +3.8% · +6.3% 다.

도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 [^6]배 근처이고, 이 배수는 파형이 아니라 **slow-time Hann 창**(프레임과 프레임 사이 축에 씌워 가장자리를 깎는 창)이 정한다.

## 같은 규약끼리 — 첫 널 대 첫 널

규약을 맞춰 **첫 널끼리** 대면 LTE·5G 에서 부호가 뒤집힌다 — 측정 주엽이 닫힌형보다 **넓다**. 세 경우 모두 `range_null_found` 가 참이다.

| 기준신호 | 측정 첫 널 | 닫힌형 $c/B_{ref}$ | 비 | 초과분 | 널 격자눈금 |
|---|---|---|---|---|---|
| WiFi VHT-LTF | 3.95 m [^12] | 3.92 m [^13] | 1.009 | +0.9% | 1.3% |
| LTE CRS | 18.00 m [^14] | 16.67 m [^15] | 1.080 | +8.0% | 0.3% |
| 5G SSB | 46.45 m [^16] | 41.64 m [^17] | 1.116 | +11.6% | 0.1% |

⚠ WiFi 의 초과분 +0.9% 은 널 탐색 격자눈금 1.3% 보다 작다 — 이 격자에서 WiFi 의 초과분은 유의하지 않다. LTE +8.0% · 5G +11.6% 는 눈금의 수십 배 밖이고, 무엇이 그 초과분을 만드는지는 이 편의 밖이다 — 다음 단계 표에 올린다.

![report03_f6_af_mainlobe](../outputs/figures/report03_f6_af_mainlobe.png)

**그림 1.** −3 dB 전폭으로 잰 주엽은 첫 널 규약의 닫힌형과 몇 % 안에서 맞는가?

## 부엽과 도플러 레플리카

주엽 밖으로 새는 에너지는 두 가지로 나타난다. **부엽**은 강한 표적이 평면 다른 곳의 약한 표적을 덮는 정도이고, **±PRF 레플리카**는 무모호 속도를 넘은 표적이 되접혀 들어오는 세기다.

표의 마지막 열 «프레임 내 시간점유» 는 **에너지가 실려 있는 표본 수 ÷ 프레임 전체 표본 수** 다(`benchmark/verify_ambiguity.py:260`). 프레임 내내 신호가 있으면 온전한 몫이 되고, 앞쪽에 뭉칠수록 작아진다.

| 기준신호 | 2D 부엽 최대 — 검출기 창 (Rb ≤ 60 m) | 같은 값 — 프레임 전체 지연축 | ±PRF 레플리카 | 프레임 내 시간점유 |
|---|---|---|---|---|
| WiFi VHT-LTF | -23.4 dB [^7] | -14.3 dB [^18] | -0.00 dB [^10] | 0.4% [^19] |
| LTE CRS | -15.0 dB [^8] | -5.3 dB [^20] | -23.27 dB [^11] | 42.9% [^21] |
| 5G SSB | -18.3 dB [^9] | -18.0 dB [^22] | -1.05 dB [^23] | 28.6% [^24] |

⚠ **두 열이 다른 것이 이 표의 요점이다.** 값도 세 표준의 순위도 지연축을 어디까지 여느냐가 정한다 — 검출기 창 안에서는 WiFi 가 가장 낮고, 프레임 전체에서는 5G 가 가장 낮다. 검출기 화면에 찍히는 것은 왼쪽 열이다. 두 창 사이의 사다리는 다음 단계 표에 올린다.

## 레플리카를 정하는 것은 점유율이 아니다

레플리카의 세기를 정하는 것은 **에너지가 프레임 안에 얼마나 퍼져 있는가**다. CRS 처럼 프레임 전체에 흩어지면 위상이 상쇄돼 레플리카가 죽고, LTF·SSB 처럼 앞쪽에 뭉치면 그대로 남는다.

이 표의 PRF 는 **검출기 프레임률**이다. 물리 주기 기준의 접힘은 [편 50 «5G SSB 는 걷는 드론에서 접힌다»](50_doppler-fold.ipynb) 가 따로 잰다 — 그 편이 같은 표를 물리 반복률로 다시 세운다.

![report03_f7_af_sidelobe](../outputs/figures/report03_f7_af_sidelobe.png)

**그림 2.** 프레임 전체 지연축에서 각 기준신호는 표적 에너지를 부엽과 도플러 레플리카에 얼마나 남기는가?

⚠ (a) 의 막대는 `psl_2d_db`·`isl_2d_db` — **프레임 한 주기 전체**의 값이다(`src/make_report03_illuminators.py:640`). 검출기 창(|Rb| ≤ 60 m) 안의 적분부엽은 WiFi -13.0 [^25] · LTE -12.4 [^26] · 5G -22.9 dB [^27] 로, 전역값(LTE +4.2 dB [^28])과 크기도 순위도 다르다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `benchmark/run_min_cell.py:74` 의 `frame_len()` 을 물리 SSB 주기(50 Hz [^29])로 확장한다 | 검출기 프레임률(2000 Hz [^30])과 40 [^31]배 벌어진 이 편의 표가 한 규약 위에 선다 | `benchmark/verify_ambiguity.py:108` |
| 거리창을 30~240 m 로 흔들어 PSL·ISL 사다리를 원장에 적는다 | 부엽 최대와 세 표준의 순위가 창의 함수인지가 수치로 확정된다 | `benchmark/verify_ambiguity.py:287` |
| 첫 널이 닫힌형보다 멀리 나오는 몫을 기준신호의 실효 점유대역으로 갈라 잰다 | LTE +8.0% · 5G +11.6% 의 초과분이 $B_{ref}$ 의 정의에서 오는지가 확정된다 | `src/waveforms.py:237` |
| 부엽 최대를 표적 두 개가 있는 장면에서 다시 잰다 | 강한 표적이 약한 표적을 덮는 거리가 수치로 확정된다 | `benchmark/verify_ambiguity.py` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 31개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report03_illuminators.json` | `detector_af_max_err_db.value` | 0.1439 |
| [^2] | `outputs/report03_illuminators.json` | `detector_af_max_err_db.n_cases` | 6 |
| [^3] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dR_ratio` | 0.895 |
| [^4] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.dR_ratio` | 0.9196 |
| [^5] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.dR_ratio` | 0.9416 |
| [^6] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dF_ratio` | 1.471 |
| [^7] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.psl_chamber_db` | -23.39 |
| [^8] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.psl_chamber_db` | -14.96 |
| [^9] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.psl_chamber_db` | -18.3 |
| [^10] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.doppler_replica_db` | -0.0002234 |
| [^11] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.doppler_replica_db` | -23.27 |
| [^12] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.range_null_m` | 3.95 |
| [^13] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dR_theory_m` | 3.916 |
| [^14] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.range_null_m` | 18 |
| [^15] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.dR_theory_m` | 16.67 |
| [^16] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.range_null_m` | 46.45 |
| [^17] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.dR_theory_m` | 41.64 |
| [^18] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.psl_2d_db` | -14.33 |
| [^19] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.ref_time_duty` | 0.004 |
| [^20] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.psl_2d_db` | -5.309 |
| [^21] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.ref_time_duty` | 0.4292 |
| [^22] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.psl_2d_db` | -18 |
| [^23] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.doppler_replica_db` | -1.053 |
| [^24] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.ref_time_duty` | 0.2865 |
| [^25] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.isl_chamber_db` | -13.01 |
| [^26] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.isl_chamber_db` | -12.44 |
| [^27] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.isl_chamber_db` | -22.91 |
| [^28] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.isl_2d_db` | 4.159 |
| [^29] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_physical_hz` | 50 |
| [^30] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_model_hz` | 2000 |
| [^31] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.ratio` | 40 |